In [1]:
# ============================================================
# CELL 1 — INSTALAÇÃO
# ============================================================
import subprocess
subprocess.run(['pip', 'install', 'encodec', '-q'], check=True)
print('✅ Dependências instaladas!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 31.9 MB/s eta 0:00:00
✅ Dependências instaladas!


In [2]:
# ============================================================
# CELL 2 — IMPORTS E SETUP
# ============================================================
import sys
import torch
import numpy as np
import soundfile as sf
import IPython.display as ipd
from pathlib import Path

# ✅ Beta: aponta pro código MS-SSM
sys.path.insert(0, '/kaggle/input/datasets/snaxcompany/nexus-code-beta/src')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


Device: cuda
PyTorch: 2.9.0+cu126
GPU: Tesla P100-PCIE-16GB


In [3]:
# ============================================================
# CELL 3 — CONFIG BETA MS-SSM
# ============================================================
CONFIG_MODEL = {
    'd_model'     : 512,
    'n_layers'    : 4,           # Beta usa 4 layers MS-SSM
    'ssm_scales'  : [16, 32, 64], # 3 escalas em paralelo
    'dropout'     : 0.1,
    'd_conv'      : 4,
    'expand'      : 1,           # expand=1 pra caber na T4
    'max_seq_len' : 750,
}

# ✅ Aponta pro checkpoint Beta S1 (ou S2 se já terminou)
# Ajuste a versão conforme o checkpoint salvo no Kaggle
CHECKPOINT_PATH = '/kaggle/input/models/destro01400/nexus-audio/pytorch/beta-1/3/checkpoints/best_session3.pt'

SAMPLE_RATE        = 24000
TOKENS_POR_SEGUNDO = 75
N_CODEBOOKS        = 8
VOCAB_SIZE         = 1024

# Janela deslizante
CHUNK_S   = 8   # dentro da janela de treino (seguro)
OVERLAP_S = 2   # contexto passado pro próximo chunk

print('✅ Config Beta MS-SSM carregada!')
print(f'   d_model: {CONFIG_MODEL["d_model"]} | scales: {CONFIG_MODEL["ssm_scales"]}')
print(f'   n_layers: {CONFIG_MODEL["n_layers"]} | dropout: {CONFIG_MODEL["dropout"]}')
print(f'   Checkpoint: {CHECKPOINT_PATH}')
print(f'   Janela deslizante: {CHUNK_S}s por chunk, {OVERLAP_S}s de overlap')


✅ Config Beta MS-SSM carregada!
   d_model: 512 | scales: [16, 32, 64]
   n_layers: 4 | dropout: 0.1
   Checkpoint: /kaggle/input/models/destro01400/nexus-audio/pytorch/beta-1/3/checkpoints/best_session3.pt
   Janela deslizante: 8s por chunk, 2s de overlap


In [4]:
# ============================================================
# CELL 4 — CARREGANDO MODELO BETA MS-SSM
# ============================================================
from model.simba_therapeutic import SiMBATherapeutic

print('Criando modelo Beta MS-SSM...')
model = SiMBATherapeutic(
    d_model     = CONFIG_MODEL['d_model'],
    n_layers    = CONFIG_MODEL['n_layers'],
    ssm_scales  = CONFIG_MODEL['ssm_scales'],
    dropout     = CONFIG_MODEL['dropout'],
    d_conv      = CONFIG_MODEL['d_conv'],
    expand      = CONFIG_MODEL['expand'],
    max_seq_len = CONFIG_MODEL['max_seq_len'],
    device      = device,
).to(device)

print(f'Carregando: {CHECKPOINT_PATH}')
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
state = ckpt.get('model_state_dict', ckpt)

# Remove prefixo 'module.' do DataParallel se necessário
new_state = {k.replace('module.', ''): v for k, v in state.items()}
model.load_state_dict(new_state, strict=False)
model.eval()

print(f'✅ Modelo Beta carregado!')
print(f'   Sessão: {ckpt.get("session", "?")} | Step: {ckpt.get("step", "?")} | Best loss: {ckpt.get("best_loss", "?")}')
print(f'   Parâmetros: {sum(p.numel() for p in model.parameters()):,}')


/kaggle/input/datasets/snaxcompany/nexus-code-beta/src/model/simba_therapeutic.py:38: UserWarning: Módulo biofeedback não disponível. Treinando sem condicionamento fisiológico.
  warnings.warn(


Criando modelo Beta MS-SSM...
Downloading: "https://dl.fbaipublicfiles.com/encodec/v0/encodec_24khz-d7cc33bc.th" to /root/.cache/torch/hub/checkpoints/encodec_24khz-d7cc33bc.th


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
100%|██████████| 88.9M/88.9M [00:00<00:00, 195MB/s]


Carregando: /kaggle/input/models/destro01400/nexus-audio/pytorch/beta-1/3/checkpoints/best_session3.pt
✅ Modelo Beta carregado!
   Sessão: 3 | Step: 25803 | Best loss: 3.6941630840301514
   Parâmetros: 39,696,110


In [5]:
# ============================================================
# CELL 5 — FUNÇÕES DE GERAÇÃO COM JANELA DESLIZANTE (VERSÃO VIP 💅✨)
# ============================================================

def sample_codebook(logits_cb, top_k=30, top_p=0.95, temperature=0.85, prev_tokens=None, rep_penalty=1.2):
    
    # 🚫 O TERROR DO DISCO ARRANHADO (Repetition Penalty)
    if prev_tokens is not None and len(prev_tokens) > 0:
        for token in torch.unique(prev_tokens):
            if logits_cb[..., token] < 0:
                logits_cb[..., token] *= rep_penalty
            else:
                logits_cb[..., token] /= rep_penalty

    # 🔥 A TEMPERATURA 
    logits_cb = logits_cb / temperature
    
    # 👑 TOP K
    if top_k > 0:
        values, _ = torch.topk(logits_cb, min(top_k, logits_cb.size(-1)))
        logits_cb = logits_cb.masked_fill(logits_cb < values[:, -1:], float('-inf'))
        
    # 👑 TOP P
    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits_cb, descending=True)
        cum_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
        remove = cum_probs > top_p
        remove[..., 1:] = remove[..., :-1].clone()
        remove[..., 0] = False
        sorted_logits[remove] = float('-inf')
        logits_cb = torch.zeros_like(logits_cb).scatter_(-1, sorted_idx, sorted_logits)
        
    return torch.multinomial(torch.softmax(logits_cb, dim=-1), 1).item()


@torch.no_grad()
def gerar_chunk(model, ctx, n_tokens, temp_inicial, temp_final, top_k, top_p, step_offset, total_tokens):
    """Gera n_tokens a partir do contexto com Efeito Drop e Repetition Penalty!"""
    generated = []
    max_ctx   = CONFIG_MODEL['max_seq_len']
    
    for i in range(n_tokens):
        # 🌡️ O CÁLCULO DO DROP (Temperatura Dinâmica)
        progresso = (step_offset + i) / total_tokens 
        temperatura_atual = temp_inicial + (progresso * (temp_final - temp_inicial))

        inp        = ctx[:, :, -max_ctx:] if ctx.shape[2] > max_ctx else ctx
        output     = model(inp)
        last_logits = output['logits'][:, :, -1, :]  # [1, 8, 1024]
        new_tok    = torch.zeros(1, N_CODEBOOKS, 1, dtype=torch.long, device=device)
        
        # 🧠 Memória para a penalidade: olha os últimos 50 tokens do codebook atual no contexto
        for cb in range(N_CODEBOOKS):
            tokens_pra_penalizar = ctx[0, cb, -50:] if ctx.shape[2] > 0 else None
            
            new_tok[0, cb, 0] = sample_codebook(
                last_logits[:, cb, :], 
                top_k, 
                top_p, 
                temperature=temperatura_atual,
                prev_tokens=tokens_pra_penalizar,
                rep_penalty=1.2
            )
            
        ctx = torch.cat([ctx, new_tok], dim=2)
        generated.append(new_tok)
        
    return torch.cat(generated, dim=2)  # [1, 8, n_tokens]


@torch.no_grad()
def gerar_audio(model, duracao_s, temp_inicial=0.60, temp_final=1.10, top_k=30, top_p=0.95, seed=42):
    """
    Geração com janela deslizante + Temperatura Dinâmica + Repetition Penalty!
    """
    torch.manual_seed(seed)
    model.eval()

    chunk_tokens   = CHUNK_S * TOKENS_POR_SEGUNDO    # 600 tokens
    overlap_tokens = OVERLAP_S * TOKENS_POR_SEGUNDO  # 150 tokens
    total_tokens   = duracao_s * TOKENS_POR_SEGUNDO
    prompt_len     = 30

    print(f'🎵 Gerando {duracao_s}s ({total_tokens} tokens)...')
    print(f'   Chunks {CHUNK_S}s | Overlap {OVERLAP_S}s | Seed {seed} | Temp {temp_inicial} -> {temp_final}')

    # Prompt: silêncio — ponto de partida limpo, sem artefatos
    ctx = torch.zeros((1, N_CODEBOOKS, prompt_len), dtype=torch.long, device=device)

    all_tokens_list = []
    tokens_gerados  = 0
    chunk_num       = 0

    while tokens_gerados < total_tokens:
        remaining   = total_tokens - tokens_gerados
        gerar_agora = min(chunk_tokens, remaining)
        chunk_num  += 1
        print(f'   Chunk {chunk_num}: {gerar_agora} tokens ({gerar_agora // TOKENS_POR_SEGUNDO:.0f}s)...', end=' ')

        chunk = gerar_chunk(model, ctx, gerar_agora, temp_inicial, temp_final, top_k, top_p, tokens_gerados, total_tokens)
        all_tokens_list.append(chunk)
        tokens_gerados += gerar_agora

        # Contexto para próximo chunk = fim do atual
        ctx = chunk[:, :, -overlap_tokens:]
        print(f'✅ ({tokens_gerados}/{total_tokens})')

    all_tokens = torch.cat(all_tokens_list, dim=2)  # [1, 8, total]
    print(f'\n✅ Tokens: {all_tokens.shape}')

    # Decodifica
    print('🔊 Decodificando...')
    encodec_model = None
    for attr in ['encodec_wrapper', 'encodec', 'codec', 'tokenizer']:
        if hasattr(model, attr):
            wrapper = getattr(model, attr)
            encodec_model = wrapper.model if hasattr(wrapper, 'model') else wrapper
            print(f'   EnCodec em model.{attr}')
            break
    if encodec_model is None:
        from encodec import EncodecModel
        encodec_model = EncodecModel.encodec_model_24khz()
        encodec_model.set_target_bandwidth(6.0)
        encodec_model = encodec_model.to(device)
        print('   EnCodec standalone carregado')

    encodec_model.eval()
    with torch.no_grad():
        audio = encodec_model.decode([(all_tokens, None)])

    audio_np = audio.squeeze().cpu().numpy()
    print(f'✅ Áudio: {len(audio_np)/SAMPLE_RATE:.2f}s @ {SAMPLE_RATE}Hz')
    return audio_np, SAMPLE_RATE

print('✅ Funções VIP prontas!')


✅ Funções VIP prontas!


In [6]:
# ============================================================
# CELL 6 — GERAR ÁUDIO VIP ← edite aqui!
# ============================================================
DURACAO_S    = 120      # segundos
TEMP_INICIAL = 0.60    # A "cama" sonora: focado e meditativo (início da música)
TEMP_FINAL   = 1.10    # O "drop": experimental e caótico (o bicho chegando!)
TOP_K        = 30
TOP_P        = 0.95
SEED         = 42      # trocar = áudio diferente

audio, sr = gerar_audio(
    model,
    duracao_s    = DURACAO_S,
    temp_inicial = TEMP_INICIAL,
    temp_final   = TEMP_FINAL,
    top_k        = TOP_K,
    top_p        = TOP_P,
    seed         = SEED
)


🎵 Gerando 120s (9000 tokens)...
   Chunks 8s | Overlap 2s | Seed 42 | Temp 0.6 -> 1.1
   Chunk 1: 600 tokens (8s)... ✅ (600/9000)
   Chunk 2: 600 tokens (8s)... ✅ (1200/9000)
   Chunk 3: 600 tokens (8s)... ✅ (1800/9000)
   Chunk 4: 600 tokens (8s)... ✅ (2400/9000)
   Chunk 5: 600 tokens (8s)... ✅ (3000/9000)
   Chunk 6: 600 tokens (8s)... ✅ (3600/9000)
   Chunk 7: 600 tokens (8s)... ✅ (4200/9000)
   Chunk 8: 600 tokens (8s)... ✅ (4800/9000)
   Chunk 9: 600 tokens (8s)... ✅ (5400/9000)
   Chunk 10: 600 tokens (8s)... ✅ (6000/9000)
   Chunk 11: 600 tokens (8s)... ✅ (6600/9000)
   Chunk 12: 600 tokens (8s)... ✅ (7200/9000)
   Chunk 13: 600 tokens (8s)... ✅ (7800/9000)
   Chunk 14: 600 tokens (8s)... ✅ (8400/9000)
   Chunk 15: 600 tokens (8s)... ✅ (9000/9000)

✅ Tokens: torch.Size([1, 8, 9000])
🔊 Decodificando...
   EnCodec em model.tokenizer
✅ Áudio: 120.00s @ 24000Hz


In [7]:
# ============================================================
# CELL 7 — SALVAR VIP 💅✨
# ============================================================
# Atualizamos o nome do arquivo pra registrar o "Drop" da temperatura!
output_path = f'/kaggle/working/nexus_VIP_{DURACAO_S}s_temp{TEMP_INICIAL}a{TEMP_FINAL}_seed{SEED}.wav'

sf.write(output_path, audio, sr)
print(f'✅ Salvo com muito luxo: {output_path}')

for w in Path('/kaggle/working').glob('*.wav'):
    print(f'   🎵 {w.name} ({w.stat().st_size // 1024} KB)')


✅ Salvo com muito luxo: /kaggle/working/nexus_VIP_120s_temp0.6a1.1_seed42.wav
   🎵 nexus_VIP_120s_temp0.6a1.1_seed42.wav (5625 KB)


In [8]:
# ============================================================
# CELL 8 — BATCH VIP: Múltiplas variações de uma vez 💅✨
# ============================================================
# Aqui a gente testa várias dinâmicas de temperatura pro bestiário!

VARIACOES = [
    # 1. O Clássico "Drop" (Suspense crescendo)
    {'seed': 42,  'temp_inicial': 0.60, 'temp_final': 1.10, 'duracao_s': 20}, 
    
    # 2. O Sombrio e Contido (Sons de caverna, drone pesado)
    {'seed': 100, 'temp_inicial': 0.50, 'temp_final': 0.85, 'duracao_s': 20}, 
    
    # 3. O Caos Absoluto (Bicho feroz, experimental)
    {'seed': 777, 'temp_inicial': 0.80, 'temp_final': 1.25, 'duracao_s': 20}, 
    
    # 4. O Inverso / Fade Out (Começa desesperador e vai acalmando)
    {'seed': 99,  'temp_inicial': 1.15, 'temp_final': 0.60, 'duracao_s': 20}  
]

print("🎧 Iniciando a festa do Batch VIP... Preparando as caixas de som!")

for i, cfg in enumerate(VARIACOES):
    print(f"\n--- 🌟 Variação {i+1}/{len(VARIACOES)}: Seed {cfg['seed']} | Temp {cfg['temp_inicial']} -> {cfg['temp_final']} ---")
    
    # Chama o nosso gerador VIP
    audio, sr = gerar_audio(
        model,
        duracao_s    = cfg['duracao_s'],
        temp_inicial = cfg['temp_inicial'],
        temp_final   = cfg['temp_final'],
        top_k        = 30,
        top_p        = 0.95,
        seed         = cfg['seed']
    )
    
    # Salva com o nome chiquérrimo já formatado pro Kaggle
    output_path = f"/kaggle/working/nexus_VIP_batch_{cfg['duracao_s']}s_t{cfg['temp_inicial']}a{cfg['temp_final']}_seed{cfg['seed']}.wav"
    sf.write(output_path, audio, sr)
    print(f"✅ Hit salvo na fila VIP: {output_path}")

print("\n🚨 BATCH FINALIZADO! Pode ir ouvir os hinos das feras! 🚨")


🎧 Iniciando a festa do Batch VIP... Preparando as caixas de som!

--- 🌟 Variação 1/4: Seed 42 | Temp 0.6 -> 1.1 ---
🎵 Gerando 20s (1500 tokens)...
   Chunks 8s | Overlap 2s | Seed 42 | Temp 0.6 -> 1.1
   Chunk 1: 600 tokens (8s)... ✅ (600/1500)
   Chunk 2: 600 tokens (8s)... ✅ (1200/1500)
   Chunk 3: 300 tokens (4s)... ✅ (1500/1500)

✅ Tokens: torch.Size([1, 8, 1500])
🔊 Decodificando...
   EnCodec em model.tokenizer
✅ Áudio: 20.00s @ 24000Hz
✅ Hit salvo na fila VIP: /kaggle/working/nexus_VIP_batch_20s_t0.6a1.1_seed42.wav

--- 🌟 Variação 2/4: Seed 100 | Temp 0.5 -> 0.85 ---
🎵 Gerando 20s (1500 tokens)...
   Chunks 8s | Overlap 2s | Seed 100 | Temp 0.5 -> 0.85
   Chunk 1: 600 tokens (8s)... ✅ (600/1500)
   Chunk 2: 600 tokens (8s)... ✅ (1200/1500)
   Chunk 3: 300 tokens (4s)... ✅ (1500/1500)

✅ Tokens: torch.Size([1, 8, 1500])
🔊 Decodificando...
   EnCodec em model.tokenizer
✅ Áudio: 20.00s @ 24000Hz
✅ Hit salvo na fila VIP: /kaggle/working/nexus_VIP_batch_20s_t0.5a0.85_seed100.wav

--- 

In [9]:
import requests

TOKEN_DO_BOT = '7781713605:AAFzHiduYdHUMAWuw56MY5VHq4LRbty4hZQ'
SEU_CHAT_ID = '6908956487'

# O texto entre aspas triplas ou parênteses deixa você organizar melhor no Python
# Olha os \n fazendo as quebras e os asteriscos fazendo o negrito!
MENSAGEM = (
    "🚨 *Alerta VIP da SnaX Company* 🚨\n\n"
    "Destro, acorda pra cuspir! 💅\n"
    "O gerador da *Nexus-Audio* acabou de finalizar lá no Kaggle."
)

url = f"https://api.telegram.org/bot{TOKEN_DO_BOT}/sendMessage"

# O truque supremo: a linha "parse_mode": "Markdown" faz o negrito funcionar!
payload = {
    "chat_id": SEU_CHAT_ID, 
    "text": MENSAGEM,
    "parse_mode": "Markdown"
}

resposta = requests.post(url, json=payload)

if resposta.status_code == 200:
    print("Aviso chique enviado pro celular do CEO! 📱✨")
else:
    print("Deu ruim:", resposta.text)


Aviso chique enviado pro celular do CEO! 📱✨
